In [1]:
!pip install pykeen -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.6 MB/s eta 0:00:00


In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate \
    bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 require

In [3]:
from pykeen.datasets import Nations
from pykeen.pipeline import pipeline
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from collections import defaultdict

In [4]:
from pykeen.datasets import UMLS
dataset = UMLS()

df_train = pd.DataFrame(
    dataset.training.triples,
    columns = ["head" , "relation","tail"]
)

df_test = pd.DataFrame(
    dataset.testing.triples,
    columns = ["head" , "relation","tail"]
)

df_all = pd.concat([df_train , df_test],ignore_index = True)

entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

print(len(df_train))
print(len(df_test))
print(len(df_all))
print(f"Entities:      {df_all['head'].nunique()}")
print(f"Relations:     {df_all['relation'].nunique()}")

Reconstructing all label-based triples. This is expensive and rarely needed.
Reconstructing all label-based triples. This is expensive and rarely needed.


5216
661
5877
Entities:      135
Relations:     46


In [5]:
result = pipeline(
    dataset="UMLS",
    model="RotatE",
    model_kwargs=dict(embedding_dim=256),
    optimizer="adam",
    optimizer_kwargs=dict(lr=1e-3),
    training_kwargs=dict(
        num_epochs=500,
        batch_size=512,
    ),
    negative_sampler="basic",
    negative_sampler_kwargs=dict(num_negs_per_pos=64),
    evaluator_kwargs=dict(filtered=True),
    random_seed=42,
)


winner_model = result.model

Training epochs on cuda:0:   0%|          | 0/500 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

...

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/11.0 [00:00<?, ?batch/s]

Evaluating on cuda:0:   0%|          | 0.00/661 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.37s seconds


In [6]:
def show_neighborhood(entity, graph):
    print(f"\n{'='*50}")
    print(f"Neighborhood of: {entity}")
    print(f"{'='*50}")
    neighbors = graph.get(entity, [])
    print(f"Direct edges: {len(neighbors)}")
    print(f"\n{'Relation':<30} {'Tail':<20}")
    print("-" * 50)
    for relation, tail in sorted(neighbors, key=lambda x: x[0]):
        print(f"{relation:<30} {tail:<20}")

In [7]:
def build_type_constraints(df):
    """
    For each relation:
      - what tail entities appear, and how often?
      - what head entities appear, and how often?
      - what (head, tail) pairs co-occur?
    
    This gives agents the UMLS ontology signal:
    "given relation R and head H, what tail types are expected?"
    """
    from collections import defaultdict
    
    # how often does each tail appear for each relation
    rel_to_tail_counts = defaultdict(lambda: defaultdict(int))
    rel_to_head_counts = defaultdict(lambda: defaultdict(int))
    rel_to_pair_counts = defaultdict(lambda: defaultdict(int))
    
    for _, row in df.iterrows():
        h, r, t = row["head"], row["relation"], row["tail"]
        rel_to_tail_counts[r][t] += 1
        rel_to_head_counts[r][h] += 1
        rel_to_pair_counts[r][(h, t)] += 1
    
    # for each (relation, head): ranked list of expected tails
    # this is the core type constraint lookup
    rel_head_to_ranked_tails = defaultdict(dict)
    for r, pair_counts in rel_to_pair_counts.items():
        # group by head
        head_to_tails = defaultdict(dict)
        for (h, t), count in pair_counts.items():
            head_to_tails[h][t] = count
        for h, tail_counts in head_to_tails.items():
            total = sum(tail_counts.values())
            rel_head_to_ranked_tails[(r, h)] = {
                t: round(c / total, 4)
                for t, c in sorted(
                    tail_counts.items(), key=lambda x: -x[1]
                )
            }
    
    # for each relation: overall tail distribution
    rel_to_tail_dist = {}
    for r, tail_counts in rel_to_tail_counts.items():
        total = sum(tail_counts.values())
        rel_to_tail_dist[r] = {
            t: round(c / total, 4)
            for t, c in sorted(
                tail_counts.items(), key=lambda x: -x[1]
            )
        }
    
    return {
        "rel_head_to_ranked_tails": dict(rel_head_to_ranked_tails),
        "rel_to_tail_dist":         rel_to_tail_dist,
        "rel_to_tail_counts":       dict(rel_to_tail_counts),
        "rel_to_head_counts":       dict(rel_to_head_counts),
    }


def get_type_constraint_signal(head, relation, true_tail,
                                predicted, constraints):
    """
    UMLS replacement for path-based only_tail_has.
    
    Returns:
      type_fit_true  - how well true_tail fits the type slot
      type_fit_pred  - how well predicted fits the type slot  
      expected_tails - top-5 expected tails for (relation, head)
      only_true_has  - relations true_tail appears in that predicted doesn't
      only_pred_has  - relations predicted appears in that true_tail doesn't
    """
    rh_tails  = constraints["rel_head_to_ranked_tails"]
    tail_dist = constraints["rel_to_tail_dist"]
    
    # type fit scores — how expected is each candidate as tail?
    specific  = rh_tails.get((relation, head), {})
    general   = tail_dist.get(relation, {})
    
    # specific (this exact head) first, fallback to general
    type_fit_true = specific.get(true_tail) or general.get(true_tail, 0.0)
    type_fit_pred = specific.get(predicted) or general.get(predicted, 0.0)
    
    # top expected tails for this (relation, head) pair
    expected_tails = list(specific.keys())[:5] or list(general.keys())[:5]
    
    # relational profile difference
    # what relations does true_tail appear in vs predicted?
    tail_counts = constraints["rel_to_tail_counts"]
    
    true_tail_rels = {
        r for r, tails in tail_counts.items()
        if true_tail in tails
    }
    pred_rels = {
        r for r, tails in tail_counts.items()
        if predicted in tails
    }
    
    only_true = true_tail_rels - pred_rels
    only_pred = pred_rels - true_tail_rels
    shared    = true_tail_rels & pred_rels
    
    return {
        "type_fit_true":  type_fit_true,
        "type_fit_pred":  type_fit_pred,
        "type_gap":       round(type_fit_true - type_fit_pred, 4),
        "expected_tails": expected_tails,
        "only_true_has":  sorted(only_true),   # ← same field name, new meaning
        "only_pred_has":  sorted(only_pred),
        "shared_rels":    sorted(shared),
    }

In [8]:
# the triples are integer IDs — convert to strings first
def build_graph(triples_array, id_to_entity, id_to_relation):
    """
    Converts integer ID triples → string adjacency dict.
    
    triples_array: numpy array of shape (N, 3) 
                   columns: [head_id, relation_id, tail_id]
    """
    graph = defaultdict(list)
    
    for h_id, r_id, t_id in triples_array:
        head     = id_to_entity[h_id]
        relation = id_to_relation[r_id]
        tail     = id_to_entity[t_id]
        graph[head].append((relation, tail))
    
    return graph

# rebuild id mappings cleanly
entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

# get raw integer arrays
train_triples = dataset.training.mapped_triples.numpy()
test_triples  = dataset.testing.mapped_triples.numpy()
all_triples   = np.vstack([train_triples, test_triples])

# build graph on training only
graph = build_graph(train_triples, id_to_entity, id_to_relation)

# also rebuild df_train and df_test correctly
df_train = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in train_triples
], columns=["head", "relation", "tail"])

df_test = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in test_triples
], columns=["head", "relation", "tail"])

df_all = pd.concat([df_train, df_test], ignore_index=True)

# verify
print(f"Train triples: {len(df_train)}")
print(f"Test triples:  {len(df_test)}")
print(f"Graph nodes:   {len(graph)}")
print(f"\nSample df_train:")
print(df_train.head(5).to_string(index=False))

Train triples: 5216
Test triples:  661
Graph nodes:   135

Sample df_train:
                head relation      tail
acquired_abnormality  affects      alga
acquired_abnormality  affects    animal
acquired_abnormality  affects  archaeon
acquired_abnormality  affects bacterium
acquired_abnormality  affects      bird


In [9]:
constraints = build_type_constraints(df_train)
print(f"Constraints built: {len(constraints['rel_to_tail_dist'])} relations")

Constraints built: 46 relations


In [10]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.cuda.empty_cache()

In [11]:
def hop_classifier(head, tail, graph, target_relation=None):
    """
    If target_relation is specified:
      single = direct edge exists with THAT relation
      multi  = reachable but via different relations
    """
    # check direct edge with matching relation
    for relation, t in graph.get(head, []):
        if t == tail:
            if target_relation is None or relation == target_relation:
                return "single", 1, relation
    
    # direct edge exists but wrong relation
    direct_wrong = [r for r, t in graph.get(head, []) if t == tail]
    if direct_wrong:
        return "multi", 1.5, f"direct but via {direct_wrong[0]}"
    
    # check 2-hop
    paths_found = []
    for r1, mid in graph.get(head, []):
        for r2, t2 in graph.get(mid, []):
            if t2 == tail:
                paths_found.append(
                    f"{head}-{r1}->{mid}-{r2}->{tail}"
                )
    
    if paths_found:
        return "multi", 2, paths_found[0]
    
    return "none", 99, []

# retest with target relation
queries = [
    ("usa", "militaryalliance", "uk"),
    ("usa", "accusation",       "cuba"),
    ("usa", "embassy",          "uk"),
    ("usa", "independence",     "china"),
]

print(f"{'Query':<45} {'Type':<8} {'Detail'}")
print("-" * 80)
for head, relation, tail in queries:
    hop_type, hops, detail = hop_classifier(
        head, tail, graph, target_relation=relation
    )
    print(f"({head}, {relation}, {tail})  "
          f"{hop_type:<8} {str(detail)[:35]}")

Query                                         Type     Detail
--------------------------------------------------------------------------------
(usa, militaryalliance, uk)  none     []
(usa, accusation, cuba)  none     []
(usa, embassy, uk)  none     []
(usa, independence, china)  none     []


In [12]:
def failure_summary(head, relation, true_tail, 
                    predicted_tail, model, df_train):
    """
    Converts failure vector → language.
    
    Tells the LLM:
    - what the model predicted
    - how many relations they share
    - what relations only the true answer has
    
    No floats. ~30 tokens.
    """
    # score both entities
    h_id    = entity_to_id[head]
    r_id    = relation_to_id[relation]
    t_id    = entity_to_id[true_tail]
    p_id    = entity_to_id[predicted_tail]
    
    score_true = model.score_hrt(
        torch.tensor([[h_id, r_id, t_id]])
    ).item()
    score_pred = model.score_hrt(
        torch.tensor([[h_id, r_id, p_id]])
    ).item()
    
    # relation analysis
    true_rels = set(
        df_train[
            (df_train["head"] == head) & 
            (df_train["tail"] == true_tail)
        ]["relation"]
    )
    pred_rels = set(
        df_train[
            (df_train["head"] == head) & 
            (df_train["tail"] == predicted_tail)
        ]["relation"]
    )
    
    shared    = true_rels & pred_rels
    only_true = true_rels - pred_rels
    only_pred = pred_rels - true_rels
    
    summary = (
        f"Model predicted '{predicted_tail}' "
        f"(score={score_pred:.3f}) over '{true_tail}' "
        f"(score={score_true:.3f}). "
        f"Shared relations: {len(shared)}. "
        f"Only '{true_tail}' has: "
        f"{', '.join(list(only_true)) if only_true else 'none'}. "
        f"Only '{predicted_tail}' has: "
        f"{', '.join(list(only_pred)) if only_pred else 'none'}."
    )
    
    return summary, {
        "score_true":  score_true,
        "score_pred":  score_pred,
        "shared":      shared,
        "only_true":   only_true,
        "only_pred":   only_pred
    }

In [13]:
def get_full_ranking_filtered(head, relation, true_tail, model):
    """
    Same as before but filters self-loop at rank 1.
    In real routing you'd never route a problem to itself.
    """
    h_id = entity_to_id[head]
    r_id = relation_to_id[relation]
    
    all_scores = []
    for i in range(len(entity_to_id)):
        # skip self-prediction
        if id_to_entity[i] == head:
            continue
        score = model.score_hrt(
            torch.tensor([[h_id, r_id, i]])
        ).item()
        all_scores.append((id_to_entity[i], score, i))
    
    all_scores.sort(key=lambda x: -x[1])
    
    ranked_entities = [e for e, s, i in all_scores]
    true_rank       = ranked_entities.index(true_tail) + 1
    predicted       = all_scores[0][0]
    
    return {
        "predicted":       predicted,
        "predicted_score": all_scores[0][1],
        "true_tail":       true_tail,
        "true_score":      [s for e,s,i in all_scores 
                           if e == true_tail][0],
        "true_rank":       true_rank,
        "full_ranking":    [(e, round(s,3)) 
                           for e,s,i in all_scores],
    }

In [14]:
# ════════════════════════════════════════════════════════
# HELD-OUT RERANKER FORMAT — UMLS version
# ════════════════════════════════════════════════════════

import json, torch, numpy as np
import torch.nn.functional as F
from collections import defaultdict
from pykeen.datasets import UMLS
from pykeen.pipeline import pipeline_from_path
import pandas as pd

# ── config ────────────────────────────────────────────────
TOPK           = 50
HELD_OUT_INPUT = r"/kaggle/input/datasets/aaryaupi/held-out/UMLS_held_out (7).json"        # preprocessed format
HELD_OUT_OUT   = "umls_bert_held_out.json"   # reranker format

# ════════════════════════════════════════════════════════
# STEP 0 — load dataset + model (reuse from training session)
# If running fresh, reload everything
# ════════════════════════════════════════════════════════

dataset = UMLS()

entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

train_triples = dataset.training.mapped_triples.numpy()
test_triples  = dataset.testing.mapped_triples.numpy()

df_train = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in train_triples
], columns=["head", "relation", "tail"])

winner_model = result.model
winner_model.eval()

# build type constraints — needed for text_input
constraints = build_type_constraints(df_train)

# build entity relation profile — needed for [DIFFERENCE] block
entity_rel_profile = defaultdict(set)
for _, row in df_train.iterrows():
    entity_rel_profile[row["tail"]].add(row["relation"])
    entity_rel_profile[row["head"]].add(row["relation"])

# cache embeddings
def get_entity_embeddings(model):
    embs = model.entity_representations[0](
        indices=None
    ).detach().cpu()
    if embs.is_complex():
        embs = embs.abs()
    return embs

EMBS = get_entity_embeddings(winner_model)
print(f"Entities: {len(entity_to_id)}  "
      f"Relations: {len(relation_to_id)}  "
      f"EMBS: {EMBS.shape}")


# ════════════════════════════════════════════════════════
# STEP 1 — top-K candidates from RotatE
# Same as Nations but batched for UMLS size
# ════════════════════════════════════════════════════════

def get_topk_candidates(head, relation, true_tail,
                         model, k=TOPK, batch_size=256):
    if (head     not in entity_to_id or
            relation not in relation_to_id):
        return []

    h_id    = entity_to_id[head]
    r_id    = relation_to_id[relation]
    all_ids = list(range(len(entity_to_id)))

    scores = {}
    for i in range(0, len(all_ids), batch_size):
        batch = all_ids[i:i + batch_size]
        hrs   = torch.tensor([[h_id, r_id, t] for t in batch])
        with torch.no_grad():
            s = model.score_hrt(hrs).squeeze(-1).cpu().tolist()
        for t_id, score in zip(batch, s):
            scores[id_to_entity[t_id]] = score

    # sort descending, skip self
    ranked = sorted(
        [(e, s) for e, s in scores.items() if e != head],
        key=lambda x: -x[1]
    )

    candidates = []
    rank        = 0
    for entity, score in ranked:
        rank += 1
        candidates.append({
            "entity":    entity,
            "kge_score": round(score, 4),
            "kge_rank":  rank,
        })
        if len(candidates) == k:
            break

    # force-insert true tail if outside top-K
    if not any(c["entity"] == true_tail for c in candidates):
        if true_tail in entity_to_id:
            t_score = scores.get(true_tail, -99.0)
            t_rank  = sum(1 for s in scores.values() if s > t_score)
            candidates.append({
                "entity":    true_tail,
                "kge_score": round(t_score, 4),
                "kge_rank":  t_rank,
            })

    return candidates


# ════════════════════════════════════════════════════════
# STEP 2 — feature extraction (UMLS version)
#
# KEY DIFFERENCE from Nations:
# Nations used direct (head, candidate) edge lookup
# UMLS uses type constraint + relational profile
# ════════════════════════════════════════════════════════

def extract_features_umls(head, relation, candidate,
                           true_tail) -> dict:
    if (head      not in entity_to_id or
            relation  not in relation_to_id or
            candidate not in entity_to_id):
        return {}

    h_id = entity_to_id[head]
    c_id = entity_to_id[candidate]

    # KGE score
    with torch.no_grad():
        kge_score = winner_model.score_hrt(
            torch.tensor([[h_id, relation_to_id[relation], c_id]])
        ).item()

    # embedding similarity
    h_vec       = EMBS[h_id]
    c_vec       = EMBS[c_id]
    sim_to_head = F.cosine_similarity(
        h_vec.unsqueeze(0), c_vec.unsqueeze(0)
    ).item()

    if true_tail in entity_to_id:
        t_vec            = EMBS[entity_to_id[true_tail]]
        sim_to_true_tail = F.cosine_similarity(
            t_vec.unsqueeze(0), c_vec.unsqueeze(0)
        ).item()
    else:
        sim_to_true_tail = 0.0

    # relational profile overlap (UMLS replacement for direct edges)
    true_rels = entity_rel_profile.get(true_tail, set())
    cand_rels = entity_rel_profile.get(candidate, set())

    shared       = true_rels & cand_rels
    only_true    = true_rels - cand_rels
    only_cand    = cand_rels - true_rels
    union        = true_rels | cand_rels
    jaccard      = len(shared) / len(union) if union else 0.0

    # type constraint signals
    rel_dist  = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = rel_dist.get(candidate, 0.0)
    ranked    = list(rel_dist.keys())
    type_rank = ranked.index(candidate) + 1 \
                if candidate in ranked else 99
    cooc      = constraints["rel_to_tail_counts"].get(
        relation, {}
    ).get(candidate, 0)

    return {
        # Nations-compatible keys (BERT trained on these)
        "kge_score":         round(kge_score,        4),
        "sim_to_head":       round(sim_to_head,       4),
        "sim_to_true_tail":  round(sim_to_true_tail,  4),
        "shared_rels_count": len(shared),
        "only_true_count":   len(only_true),
        "only_cand_count":   len(only_cand),
        "jaccard_overlap":   round(jaccard,           4),
        "direct_edge":       int(tf > 0.0),
        "hop_count":         1,
        # UMLS extras
        "type_fit":          round(tf,                4),
        "type_rank":         type_rank,
        "cooc_count":        cooc,
    }


# ════════════════════════════════════════════════════════
# STEP 3 — text_input builder (UMLS format)
# Must match exactly what BERT was trained on
# ════════════════════════════════════════════════════════

def build_text_input_held_out(head, relation, candidate,
                               features, true_tail=None):
    """
    Identical to build_text_input_umls used during training.
    MUST match — BERT scores based on this format.
    """
    f         = features
    sig       = constraints["rel_to_tail_dist"].get(relation, {})
    tf        = sig.get(candidate, 0.0)
    ranked    = list(sig.keys())
    type_rank = ranked.index(candidate) + 1 \
                if candidate in ranked else 99
    expected  = ", ".join(ranked[:3]) if ranked else "none"
    cand_rels = sorted(
        entity_rel_profile.get(candidate, set())
    )[:5]

    text = (
        f"[QUERY]\n{head} | {relation} | ?\n\n"
        f"[CANDIDATE]\n{candidate}\n\n"
        f"[TYPE CONSTRAINTS]\n"
        f"type_fit = {tf:.4f}\n"
        f"type_rank = {type_rank}\n"
        f"expected_tails = {expected}\n\n"
        f"[KGE SIGNALS]\n"
        f"kge_score = {f.get('kge_score', 0.0):.3f}\n\n"
        f"[RELATIONAL PROFILE]\n"
        f"candidate_appears_in = "
        f"{', '.join(cand_rels) if cand_rels else 'none'}\n"
    )

    return text


# ════════════════════════════════════════════════════════
# STEP 4 — convert one held-out record
#
# held-out record is in preprocessed format:
# {head, relation, tail, true_rank, subgraph,
#  only_tail_has, fail_summary, hard_failure, ...}
# ════════════════════════════════════════════════════════

def convert_held_out_record(record) -> dict | None:
    head      = record["head"]
    relation  = record["relation"]
    true_tail = record["tail"]

    if not all(e in entity_to_id for e in [head, true_tail]):
        return None
    if relation not in relation_to_id:
        return None

    # get top-K candidates from RotatE
    raw = get_topk_candidates(
        head, relation, true_tail, winner_model, k=TOPK
    )
    if not raw:
        return None

    # reuse precomputed fields from preprocessed record
    only_tail_has = record.get("only_tail_has", [])
    fail_summary  = record.get("fail_summary",  "")
    sim_head_str  = record.get("sim_head",       "")

    # subgraph string
    raw_subgraph  = record.get("subgraph", [])
    oth_set       = set(only_tail_has)
    subgraph_str  = "\n".join(
        f"  {h} --{r}--> {t}" + (" ◆" if r in oth_set else "")
        for h, r, t in raw_subgraph[:12]
    )

    # build candidates with features + text_input
    candidates_out = []
    for c in raw:
        entity   = c["entity"]
        features = extract_features_umls(
            head, relation, entity, true_tail
        )
        text_input = build_text_input_held_out(
            head, relation, entity,
            features  = features,
            true_tail = true_tail if entity != true_tail else None
        )
        candidates_out.append({
            "entity":     entity,
            "label":      1 if entity == true_tail else 0,
            "soft_label": None,
            "neg_type":   "true" if entity == true_tail
                          else "model_topk",
            "kge_rank":   c["kge_rank"],
            "text_input": text_input,
            "features":   features,
        })

    # sort by kge_score descending
    candidates_out.sort(
        key=lambda c: -(c["features"].get("kge_score") or -99)
    )

    rotate_rank = next(
        (i + 1 for i, c in enumerate(candidates_out)
         if c["entity"] == true_tail), None
    )

    return {
        "triple":              [head, relation, true_tail],
        "true_rank":           record.get("true_rank", -1),
        "hop_type":            record.get("hop_type", "multi"),
        "hard_failure":        record.get("hard_failure", False),
        "agent_confidence":    0.5,
        "rotate_rank_in_topk": rotate_rank,
        "candidates":          candidates_out,
        "context": {
            "subgraph_str":  subgraph_str,
            "only_tail_has": only_tail_has,
            "fail_summary":  fail_summary,
            "sim_head":      sim_head_str,
        }
    }


# ════════════════════════════════════════════════════════
# STEP 5 — run
# ════════════════════════════════════════════════════════

with open(HELD_OUT_INPUT) as f:
    held_out_records = json.load(f)

print(f"Total held-out: {len(held_out_records)}")

reranker_held_out, errors = [], 0

for i, record in enumerate(held_out_records):
    try:
        out = convert_held_out_record(record)
        if out is not None:
            reranker_held_out.append(out)
    except Exception as e:
        print(f"[ERROR] {i}: {e}")
        errors += 1

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(held_out_records)} done")

with open(HELD_OUT_OUT, "w") as f:
    json.dump(reranker_held_out, f, indent=2)

# ── stats ──────────────────────────────────────────────────
print(f"\nOutput : {len(reranker_held_out)} records  "
      f"({errors} errors)")

missing = sum(
    1 for r in reranker_held_out
    if not any(c["entity"] == r["triple"][2]
               for c in r["candidates"])
)
print(f"Missing true tail in candidates: {missing}")

rotate_ranks = [
    r["rotate_rank_in_topk"]
    for r in reranker_held_out
    if r["rotate_rank_in_topk"] is not None
]
hits1 = sum(1 for r in rotate_ranks if r == 1)
hits3 = sum(1 for r in rotate_ranks if r <= 3)

print(f"\nRotatE baseline within top-{TOPK}:")
print(f"  Hits@1 : {hits1}/{len(rotate_ranks)}"
      f"  ({100*hits1/max(len(rotate_ranks),1):.1f}%)")
print(f"  Hits@3 : {hits3}/{len(rotate_ranks)}"
      f"  ({100*hits3/max(len(rotate_ranks),1):.1f}%)")

print(f"\nType constraint coverage:")
has_type_fit = sum(
    1 for r in reranker_held_out
    for c in r["candidates"]
    if c["features"].get("type_fit", 0) > 0
)
total_cands = sum(len(r["candidates"]) for r in reranker_held_out)
print(f"  Candidates with type_fit > 0: "
      f"{has_type_fit}/{total_cands} "
      f"({100*has_type_fit/max(total_cands,1):.1f}%)")

print(f"\nSample positive text_input:")
pos = next(
    c for c in reranker_held_out[0]["candidates"]
    if c["label"] == 1
)
print(pos["text_input"])

print(f"\nSample negative text_input:")
neg = next(
    c for c in reranker_held_out[0]["candidates"]
    if c["label"] == 0
)
print(neg["text_input"])

print(f"\nSaved → {HELD_OUT_OUT}")

Entities: 135  Relations: 46  EMBS: torch.Size([135, 256])
Total held-out: 331
  25/331 done
  50/331 done
  75/331 done
  100/331 done
  125/331 done
  150/331 done
  175/331 done
  200/331 done
  225/331 done
  250/331 done
  275/331 done
  300/331 done
  325/331 done

Output : 331 records  (0 errors)
Missing true tail in candidates: 0

RotatE baseline within top-50:
  Hits@1 : 3/331  (0.9%)
  Hits@3 : 18/331  (5.4%)

Type constraint coverage:
  Candidates with type_fit > 0: 6781/16552 (41.0%)

Sample positive text_input:
[QUERY]
steroid | issue_in | ?

[CANDIDATE]
biomedical_occupation_or_discipline

[TYPE CONSTRAINTS]
type_fit = 0.4843
type_rank = 2
expected_tails = occupation_or_discipline, biomedical_occupation_or_discipline

[KGE SIGNALS]
kge_score = -8.634

[RELATIONAL PROFILE]
candidate_appears_in = associated_with, isa, issue_in, method_of, practices


Sample negative text_input:
[QUERY]
steroid | issue_in | ?

[CANDIDATE]
occupation_or_discipline

[TYPE CONSTRAINTS]
type_fit

In [15]:
"""
# ════════════════════════════════════════════════════════
# HELD-OUT RERANKER FORMAT — full standalone notebook
# ════════════════════════════════════════════════════════

import json, torch, numpy as np
import torch.nn.functional as F
from collections import defaultdict
from pykeen.datasets import Nations
from pykeen.pipeline import pipeline
import pandas as pd

# ── config ────────────────────────────────────────────────
TOPK           = 10
HELD_OUT_INPUT = "/kaggle/input/datasets/aaryaupi/held-out/nations_held_out.json"
HELD_OUT_OUT   = "bert_reranker_held_out.json"

# ════════════════════════════════════════════════════════
# STEP 0 — load dataset + train model
# ════════════════════════════════════════════════════════

dataset = Nations()

entity_to_id   = dataset.training.entity_to_id
relation_to_id = dataset.training.relation_to_id
id_to_entity   = {v: k for k, v in entity_to_id.items()}
id_to_relation = {v: k for k, v in relation_to_id.items()}

train_triples = dataset.training.mapped_triples.numpy()
test_triples  = dataset.testing.mapped_triples.numpy()

df_train = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in train_triples
], columns=["head", "relation", "tail"])

df_test = pd.DataFrame([
    (id_to_entity[h], id_to_relation[r], id_to_entity[t])
    for h, r, t in test_triples
], columns=["head", "relation", "tail"])


def build_graph(triples_array, id_to_entity, id_to_relation):
    graph = defaultdict(list)
    for h_id, r_id, t_id in triples_array:
        graph[id_to_entity[h_id]].append(
            (id_to_relation[r_id], id_to_entity[t_id])
        )
    return graph

graph = build_graph(train_triples, id_to_entity, id_to_relation)

# train RotatE — use seed 123 to match winner_model
result = pipeline(
    dataset="Nations",
    model="RotatE",
    training_kwargs=dict(num_epochs=200),
    random_seed=123,
)
winner_model = result.model
winner_model.eval()

print(f"MRR: {result.get_metric('mrr'):.4f}")


def get_entity_embeddings(model):
    embs = model.entity_representations[0](
        indices=None
    ).detach().cpu()
    if embs.is_complex():
        embs = embs.abs()
    return embs

EMBS = get_entity_embeddings(winner_model)

print(f"Entities: {len(entity_to_id)}  "
      f"Relations: {len(relation_to_id)}  "
      f"EMBS: {EMBS.shape}")


# ════════════════════════════════════════════════════════
# STEP 1 — helper functions
# ════════════════════════════════════════════════════════

def hop_classifier(head, tail, graph, target_relation=None):
    for relation, t in graph.get(head, []):
        if t == tail:
            if target_relation is None or relation == target_relation:
                return "single", 1, relation
    direct_wrong = [r for r, t in graph.get(head, []) if t == tail]
    if direct_wrong:
        return "multi", 1.5, f"direct but via {direct_wrong[0]}"
    paths_found = []
    for r1, mid in graph.get(head, []):
        for r2, t2 in graph.get(mid, []):
            if t2 == tail:
                paths_found.append(
                    f"{head}-{r1}->{mid}-{r2}->{tail}"
                )
    if paths_found:
        return "multi", 2, paths_found[0]
    return "none", 99, []


def format_candidate_structured(head, relation, candidate,
                                  features, only_tail_has) -> str:
    f    = features
    diff = " | ".join(only_tail_has) if only_tail_has else "none"
    return "\n\n".join([
        f"[QUERY]\n{head} | {relation} | ?",
        f"[CANDIDATE]\n{candidate}",
        (f"[KEY SIGNALS]\n"
         f"kge_score = {f.get('kge_score', 0.0):.3f}\n"
         f"sim_to_head = {f.get('sim_to_head', 0.0):.2f}\n"
         f"shared_rels = {f.get('shared_rels_count', 0)}\n"
         f"only_true = {f.get('only_true_count', 0)}\n"
         f"only_cand = {f.get('only_cand_count', 0)}\n"
         f"direct_edge = {f.get('direct_edge', 0)}"),
        f"[DIFFERENCE]\nonly_true_has = {diff}",
    ])


# ════════════════════════════════════════════════════════
# STEP 2 — RotatE top-K live query
# ════════════════════════════════════════════════════════

def get_topk_candidates(head, relation, true_tail,
                        model, k=TOPK):
    if (head not in entity_to_id or
            relation not in relation_to_id):
        return []

    h_id = entity_to_id[head]
    r_id = relation_to_id[relation]
    n    = len(entity_to_id)

    with torch.no_grad():
        scores = model.score_hrt(torch.stack([
            torch.full((n,), h_id, dtype=torch.long),
            torch.full((n,), r_id, dtype=torch.long),
            torch.arange(n,        dtype=torch.long),
        ], dim=1)).squeeze(-1).cpu().numpy()

    order      = np.argsort(scores)[::-1]
    candidates = []
    rank        = 0

    for idx in order:
        entity = id_to_entity[idx]
        if entity == head:
            continue
        rank += 1
        candidates.append({
            "entity":    entity,
            "kge_score": float(round(scores[idx], 4)),
            "kge_rank":  rank,
        })
        if len(candidates) == k:
            break

    # force-insert true tail if outside top-K
    if not any(c["entity"] == true_tail for c in candidates):
        if true_tail in entity_to_id:
            t_idx   = entity_to_id[true_tail]
            t_score = float(scores[t_idx])
            t_rank  = int((scores > t_score).sum())
            candidates.append({
                "entity":    true_tail,
                "kge_score": round(t_score, 4),
                "kge_rank":  t_rank,
            })

    return candidates


# ════════════════════════════════════════════════════════
# STEP 3 — feature extraction (identical to training)
# ════════════════════════════════════════════════════════

def extract_features_infer(head, relation, candidate,
                            true_tail) -> dict:
    if (head      not in entity_to_id or
            relation  not in relation_to_id or
            candidate not in entity_to_id):
        return {}

    h_id = entity_to_id[head]
    r_id = relation_to_id[relation]
    c_id = entity_to_id[candidate]

    with torch.no_grad():
        kge_score = winner_model.score_hrt(
            torch.tensor([[h_id, r_id, c_id]])
        ).item()

    h_vec = EMBS[h_id]
    c_vec = EMBS[c_id]
    sim_to_head = F.cosine_similarity(
        h_vec.unsqueeze(0), c_vec.unsqueeze(0)
    ).item()

    if true_tail in entity_to_id:
        t_vec = EMBS[entity_to_id[true_tail]]
        sim_to_true = F.cosine_similarity(
            t_vec.unsqueeze(0), c_vec.unsqueeze(0)
        ).item()
    else:
        sim_to_true = 0.0

    cand_rels = set(df_train[
        (df_train["head"] == head) &
        (df_train["tail"] == candidate)
    ]["relation"])
    true_rels = set(df_train[
        (df_train["head"] == head) &
        (df_train["tail"] == true_tail)
    ]["relation"])

    shared_count = len(cand_rels & true_rels)
    only_cand    = len(cand_rels - true_rels)
    only_true    = len(true_rels - cand_rels)
    total_union  = len(cand_rels | true_rels)
    jaccard      = shared_count / total_union if total_union > 0 else 0.0

    direct_edge = any(
        r == relation and t == candidate
        for r, t in graph.get(head, [])
    )
    hop_type, hop_count, _ = hop_classifier(
        head, candidate, graph, target_relation=relation
    )

    return {
        "kge_score":         round(kge_score,   4),
        "sim_to_head":       round(sim_to_head, 4),
        "sim_to_true_tail":  round(sim_to_true, 4),
        "shared_rels_count": shared_count,
        "only_cand_count":   only_cand,
        "only_true_count":   only_true,
        "jaccard_overlap":   round(jaccard,      4),
        "direct_edge":       int(direct_edge),
        "hop_type":          hop_type,
        "hop_count":         hop_count,
    }


# ════════════════════════════════════════════════════════
# STEP 4 — only_tail_has per candidate
# ════════════════════════════════════════════════════════

def get_only_tail_has_for_candidate(head, true_tail,
                                     candidate) -> list:
    true_rels = set(df_train[
        (df_train["head"] == head) &
        (df_train["tail"] == true_tail)
    ]["relation"])
    cand_rels = set(df_train[
        (df_train["head"] == head) &
        (df_train["tail"] == candidate)
    ]["relation"])
    return sorted(true_rels - cand_rels)


# ════════════════════════════════════════════════════════
# STEP 5 — convert one held-out record
# ════════════════════════════════════════════════════════

def convert_held_out_record(record) -> dict | None:
    head      = record["head"]
    relation  = record["relation"]
    true_tail = record["tail"]

    if not all(e in entity_to_id for e in [head, true_tail]):
        return None
    if relation not in relation_to_id:
        return None

    raw = get_topk_candidates(
        head, relation, true_tail, winner_model, k=TOPK
    )
    if not raw:
        return None

    predicted = next(
        (c["entity"] for c in raw if c["entity"] != true_tail),
        raw[0]["entity"]
    )

    # reuse already-computed fields — no need to recompute
    record_only_tail_has = record.get("only_tail_has", [])
    fail_sum             = record.get("fail_summary",  "")
    sim_head_str         = record.get("sim_head",      "")

    # subgraph: stored as list of triples → convert to string
    raw_subgraph = record.get("subgraph", [])
    oth_set      = set(record_only_tail_has)
    subgraph_str = "\n".join(
        f"  {h} --{r}--> {t}" + (" \u25c6" if r in oth_set else "")
        for h, r, t in raw_subgraph[:12]
    )

    # build candidates
    candidates_out = []
    for c in raw:
        entity   = c["entity"]
        features = extract_features_infer(
            head, relation, entity, true_tail
        )
        tail_has = (
            record_only_tail_has if entity == true_tail
            else get_only_tail_has_for_candidate(
                head, true_tail, entity
            )
        )
        text_input = format_candidate_structured(
            head, relation, entity,
            features      = features,
            only_tail_has = tail_has,
        )
        candidates_out.append({
            "entity":     entity,
            "label":      1 if entity == true_tail else 0,
            "soft_label": None,
            "neg_type":   "true" if entity == true_tail
                          else "model_topk",
            "kge_rank":   c["kge_rank"],
            "text_input": text_input,
            "features":   features,
        })

    candidates_out.sort(
        key=lambda c: -(c["features"].get("kge_score") or -99)
    )

    rotate_rank = next(
        (i + 1 for i, c in enumerate(candidates_out)
         if c["entity"] == true_tail), None
    )

    return {
        "triple":              [head, relation, true_tail],
        "true_rank":           record.get("true_rank", -1),
        "hop_type":            record.get("hop_type", "multi"),
        "hard_failure":        record.get("hard_failure", False),
        "agent_confidence":    0.5,
        "rotate_rank_in_topk": rotate_rank,
        "candidates":          candidates_out,
        "context": {
            "subgraph_str":  subgraph_str,
            "only_tail_has": record_only_tail_has,
            "fail_summary":  fail_sum,
            "sim_head":      sim_head_str,
        }
    }


# ════════════════════════════════════════════════════════
# STEP 6 — run
# ════════════════════════════════════════════════════════

with open(HELD_OUT_INPUT) as f:
    held_out_records = json.load(f)

print(f"Total held-out: {len(held_out_records)}")

reranker_held_out, errors = [], 0

for i, record in enumerate(held_out_records):
    try:
        out = convert_held_out_record(record)
        if out is not None:
            reranker_held_out.append(out)
    except Exception as e:
        print(f"[ERROR] {i}: {e}")
        errors += 1

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(held_out_records)} done")

with open(HELD_OUT_OUT, "w") as f:
    json.dump(reranker_held_out, f, indent=2)

# ── stats ─────────────────────────────────────────────────
print(f"\nOutput: {len(reranker_held_out)} records  ({errors} errors)")

missing = sum(
    1 for r in reranker_held_out
    if not any(c["entity"] == r["triple"][2]
               for c in r["candidates"])
)
print(f"Missing true tail in candidates: {missing}")

rotate_ranks = [r["rotate_rank_in_topk"] for r in reranker_held_out
                if r["rotate_rank_in_topk"] is not None]
hits1 = sum(1 for r in rotate_ranks if r == 1)
hits3 = sum(1 for r in rotate_ranks if r <= 3)
print(f"\nRotatE baseline within top-{TOPK}:")
print(f"  Hits@1: {hits1}/{len(rotate_ranks)}"
      f"  ({100*hits1/max(len(rotate_ranks),1):.1f}%)")
print(f"  Hits@3: {hits3}/{len(rotate_ranks)}"
      f"  ({100*hits3/max(len(rotate_ranks),1):.1f}%)")

print(f"\nSample positive text_input:")
pos = next(c for c in reranker_held_out[0]["candidates"]
           if c["label"] == 1)
print(pos["text_input"])

print(f"\nSample negative text_input:")
neg = next(c for c in reranker_held_out[0]["candidates"]
           if c["label"] == 0)
print(neg["text_input"])

print(f"\nSaved → {HELD_OUT_OUT}")
"""

'\n# ════════════════════════════════════════════════════════\n# HELD-OUT RERANKER FORMAT — full standalone notebook\n# ════════════════════════════════════════════════════════\n\nimport json, torch, numpy as np\nimport torch.nn.functional as F\nfrom collections import defaultdict\nfrom pykeen.datasets import Nations\nfrom pykeen.pipeline import pipeline\nimport pandas as pd\n\n# ── config ────────────────────────────────────────────────\nTOPK           = 10\nHELD_OUT_INPUT = "/kaggle/input/datasets/aaryaupi/held-out/nations_held_out.json"\nHELD_OUT_OUT   = "bert_reranker_held_out.json"\n\n# ════════════════════════════════════════════════════════\n# STEP 0 — load dataset + train model\n# ════════════════════════════════════════════════════════\n\ndataset = Nations()\n\nentity_to_id   = dataset.training.entity_to_id\nrelation_to_id = dataset.training.relation_to_id\nid_to_entity   = {v: k for k, v in entity_to_id.items()}\nid_to_relation = {v: k for k, v in relation_to_id.items()}\n\nt